# カワウソ黒板解説 ／ 動画レンダリング

台本を書き換えて上から実行すると **MP4 が出ます。**

**先にアップロードするもの**（左のファイルペインにドラッグ）
- `render.py` `mkpng.py`
- キャラ画像5枚 … `char_surprise.png` `char_explain.png` `char_serious.png` `char_proud.png` `char_blink.png`
  （透過PNG推奨。無ければグレーの代役が入るので、まず流れを確認できます）
- `narration.wav` … ナレーション音声（作り方は手順6）
- `bgm.mp3` … 任意

**やらないこと**
- キャラ画像の生成はここではしません。一貫性がいま使っているツールに劣るので、絵は外で作ってください
- 投稿の自動化はしません。**1日1本、手で出す**（YouTubeの量産判定を避けるため）


## 1. 準備

In [ ]:
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!pip -q install pillow numpy
!ffmpeg -version | head -1
import os
os.makedirs('assets', exist_ok=True)
print('OK')

## 2. アップロードしたキャラ画像を assets/ に移す

In [ ]:
import glob, shutil, os
for f in glob.glob('char_*.png'):
    shutil.move(f, 'assets/' + os.path.basename(f))
print(sorted(os.listdir('assets')))

## 3. 黒板の図をつくる

`mkpng.py` が水分子・衝突・氷の格子を透過PNGで書き出します。
新しい図が必要になったら、このファイルに関数を足すだけです。

In [ ]:
!python3 mkpng.py
from IPython.display import Image as IPImage, display
display(IPImage('assets/01_water_molecule.png', width=200))

## 4. 台本

**毎回ここだけ書き換えます。** `render.py` の `SCRIPT` を上書きしています。

- `end` … そのカットが終わる秒
- `face` … 使うキャラ画像（surprise / explain / serious / proud）
- `pointer` … 指し棒で黒板を叩く秒（省略可）
- `items` … 黒板に出るもの。`t` は出現秒
  - `text` … 文字（`\n` で改行、`color=MUSTARD` で黄色）
  - `fig` … 図のPNG、`scale` は黒板幅に対する比、`spin` を書くと回転（秒/1回転）

In [ ]:
import render
from render import MUSTARD

render.SCRIPT = [
    dict(end=3.5,  face='surprise', items=[
        dict(t=0.4, text='電子レンジは\n何を温めてる？', size=84)]),

    dict(end=9.5,  face='explain', items=[
        dict(t=3.9, text='食べ物じゃない', size=68),
        dict(t=6.6, text='水', size=190, color=MUSTARD)]),

    dict(end=20.0, face='serious', pointer=17.8, items=[
        dict(t=9.9,  fig='01_water_molecule.png', scale=.52, spin=1.1),
        dict(t=17.8, text='1秒に24億回', size=104, color=MUSTARD)]),

    dict(end=28.0, face='explain', pointer=23.6, items=[
        dict(t=20.4, fig='02_collision.png', scale=.86),
        dict(t=23.6, text='ぶつかる　→　熱', size=72, color=MUSTARD)]),

    dict(end=35.5, face='explain', items=[
        dict(t=28.4, text='お皿は温まらない', size=80)]),

    dict(end=46.0, face='proud', pointer=38.4, items=[
        dict(t=36.0, text='温めてない', size=68),
        dict(t=38.4, text='水を暴れさせてる', size=84, color=MUSTARD)]),
]
render.DURATION = render.SCRIPT[-1]['end']
print('尺 %.1f 秒 / %d カット' % (render.DURATION, len(render.SCRIPT)))

## 5. まず1枚だけ見て確認する

全フレーム書き出す前に、要所だけ見ます。おかしければ手順4に戻る。

In [ ]:
from IPython.display import display
for t in [1.0, 7.0, 13.0, 24.0, 40.0]:
    im = render.render_frame(t)
    im.thumbnail((260, 460))
    print('t = %.1fs' % t); display(im)

## 6. ナレーション音声

**A案（かんたん・おすすめ）** CapCutかVOICEVOXアプリで読み上げを作り、`narration.wav` としてアップロード。

**B案** 下のセルでVOICEVOXエンジンをColab上に立てて合成する。初回は数分かかります。

In [ ]:
# B案：VOICEVOX（使わないならこのセルは飛ばす）
# !wget -q https://github.com/VOICEVOX/voicevox_engine/releases/latest/download/voicevox_engine-linux-cpu.7z.001
# ... エンジンを展開して起動し、http://localhost:50021 に POST する
# 動かない場合はA案にしてください（音声だけ外で作れば残りは全部自動です）
import os
print('narration.wav:', '有り' if os.path.exists('narration.wav') else '無し（無音で書き出します）')

## 7. 全フレーム書き出し

46秒 × 30fps = 約1400枚。数分かかります。

In [ ]:
import os, time
os.makedirs('frames', exist_ok=True)
n = int(render.DURATION * render.FPS)
t0 = time.time()
for i in range(n):
    render.render_frame(i / render.FPS).save('frames/%05d.png' % i)
    if i % 150 == 0:
        print('  %d / %d  (%.0fs)' % (i, n, time.time() - t0), flush=True)
print('完了 %d 枚 / %.0f秒' % (n, time.time() - t0))

## 8. MP4に合成

字幕は `ep01.srt` から焼き込みます（アップロードしておく）。
BGMは `bgm.mp3` があれば音量20%で混ぜます。

In [ ]:
import os, subprocess, shlex

FPS = render.FPS
has_srt = os.path.exists('ep01.srt')
has_wav = os.path.exists('narration.wav')
has_bgm = os.path.exists('bgm.mp3')

vf = []
if has_srt:
    vf.append("subtitles=ep01.srt:force_style='FontName=IPAPGothic,FontSize=17,"
              "PrimaryColour=&H00FFFFFF&,OutlineColour=&H90000000&,BorderStyle=1,"
              "Outline=2,Shadow=0,Alignment=2,MarginV=430'")

cmd = ['ffmpeg', '-y', '-framerate', str(FPS), '-i', 'frames/%05d.png']
if has_wav: cmd += ['-i', 'narration.wav']
if has_bgm: cmd += ['-i', 'bgm.mp3']

if has_wav and has_bgm:
    cmd += ['-filter_complex', '[2:a]volume=0.20[b];[1:a][b]amix=inputs=2:duration=first[a]',
            '-map', '0:v', '-map', '[a]']
elif has_wav:
    cmd += ['-map', '0:v', '-map', '1:a']

if vf: cmd += ['-vf', ','.join(vf)]
cmd += ['-c:v', 'libx264', '-preset', 'medium', '-crf', '19',
        '-pix_fmt', 'yuv420p', '-r', str(FPS), '-shortest', 'ep01.mp4']

print(' '.join(shlex.quote(c) for c in cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stderr[-1200:] if r.returncode else '書き出し成功 → ep01.mp4')

## 9. 確認してダウンロード

In [ ]:
from IPython.display import HTML
import base64, os
print('%.1f MB' % (os.path.getsize('ep01.mp4') / 1e6))
b = base64.b64encode(open('ep01.mp4','rb').read()).decode()
display(HTML('<video controls style="max-height:70vh" src="data:video/mp4;base64,%s"></video>' % b))

In [ ]:
from google.colab import files
files.download('ep01.mp4')

---

## 2本目以降

手順 **4 → 5 → 7 → 8 → 9** だけ。台本のdictを書き換えて実行するだけです。
キャラ画像と図は使い回せるので、人間の作業は台本を書く時間だけになります。

**投稿は手で。1日1本まで。**
